In [ ]:
import requests
import re
import time
import pandas as pd

MAX_RETRIES = 5
RETRY_BACKOFF = [2, 5, 10, 20, 30]  # seconds to wait between retries

# ── Keywords ──────────────────────────────────────────────────────────

LLM_CHOICES = [
    "chatbot",
    "chatbots",
    "artificial intelligence",
    "generative AI",
    "GenAI",
    "large language model",
    "large language models",
    "LLM",
    "LLMs",
]

HEALTH_OUTCOMES = [
    "loneliness",
    "social isolation",
    "social alienation",
    "social exclusion",
    "social withdrawal",
    "social connection",
    "social inclusion",
    "belonging",
    "social satisfaction",
    "perceived cohesion",
    "social well-being",
    "social wellbeing",
    "interpersonal well-being",
    "interpersonal wellbeing",
]

# ── Utility ───────────────────────────────────────────────────────────────────

def is_multi_word(term):
    return len(term.strip().split()) > 1

def get_anchor(term):
    """Single-word terms are their own anchor. Multi-word: use last word."""
    if is_multi_word(term):
        return term.strip().split()[-1].lower()
    return term.strip().lower()

def exact_match(text, term):
    """Check full exact phrase in text, case-insensitive, whole word."""
    pattern = r'\b' + re.escape(term.lower()) + r'\b'
    return bool(re.search(pattern, text.lower()))

def get_searchable_text(paper):
    attrs = paper.get("attributes", {})
    title = attrs.get("title", "") or ""
    description = attrs.get("description", "") or ""
    tags = " ".join(attrs.get("tags", []))
    return f"{title} {description} {tags}"

def get_contributors(paper_id):
    print(f"    [contributors] Fetching for {paper_id}...")
    url = f"https://api.osf.io/v2/preprints/{paper_id}/contributors/"
    response = requests.get(url)
    if response.status_code != 200:
        return []
    contributors = []
    for c in response.json().get("data", []):
        name = (c.get("embeds", {})
                 .get("users", {})
                 .get("data", {})
                 .get("attributes", {})
                 .get("full_name"))
        if not name:
            name = c.get("attributes", {}).get("full_name", "Unknown")
        contributors.append(name)
    return contributors

def extract_info(paper):
    attrs = paper.get("attributes", {})
    links = paper.get("links", {})
    rels = paper.get("relationships", {})
    subjects = []
    for path in attrs.get("subjects", []):
        if path:
            subjects.append(" > ".join(s["text"] for s in path))
    return {
        "id": paper.get("id"),
        "title": attrs.get("title"),
        "description": attrs.get("description"),
        "date_published": attrs.get("date_published", "")[:10],
        "doi": links.get("preprint_doi") or attrs.get("doi"),
        "url": links.get("html"),
        "tags": attrs.get("tags", []),
        "subjects": subjects,
        "reviews_state": attrs.get("reviews_state"),
        "provider": rels.get("provider", {}).get("data", {}).get("id"),
    }

# ── Core fetch ────────────────────────────────────────────────────────────────

def fetch_by_anchor(anchor):
    """
    Fetch all SocArXiv papers whose description contains the anchor word.
    Retries on 502/5xx errors with exponential backoff.
    On persistent failure mid-pagination, logs the failed page but keeps
    what was already collected rather than stopping entirely.
    """
    print(f"    [api] Fetching anchor '{anchor}'...")
    url = "https://api.osf.io/v2/preprints/"
    params = {
        "provider": "socarxiv",
        "filter[description]": anchor,
        "page[size]": 100
    }
    results = []
    page = 1
    failed_pages = []

    while url:
        success = False
        last_error = None

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = requests.get(url, params=params, timeout=30)

                if response.status_code == 200:
                    success = True
                    break

                elif response.status_code in (502, 503, 504):
                    wait = RETRY_BACKOFF[min(attempt - 1, len(RETRY_BACKOFF) - 1)]
                    print(f"    [warn] Status {response.status_code} for '{anchor}' "
                          f"page {page}, attempt {attempt}/{MAX_RETRIES}. "
                          f"Retrying in {wait}s...")
                    time.sleep(wait)
                    last_error = response.status_code

                else:
                    print(f"    [error] Non-retryable status {response.status_code} "
                          f"for '{anchor}' page {page}. Stopping pagination.")
                    return results

            except requests.exceptions.Timeout:
                wait = RETRY_BACKOFF[min(attempt - 1, len(RETRY_BACKOFF) - 1)]
                print(f"    [warn] Timeout for '{anchor}' page {page}, "
                      f"attempt {attempt}/{MAX_RETRIES}. Retrying in {wait}s...")
                time.sleep(wait)
                last_error = "timeout"

            except requests.exceptions.ConnectionError as e:
                wait = RETRY_BACKOFF[min(attempt - 1, len(RETRY_BACKOFF) - 1)]
                print(f"    [warn] Connection error for '{anchor}' page {page}, "
                      f"attempt {attempt}/{MAX_RETRIES}. Retrying in {wait}s...")
                time.sleep(wait)
                last_error = str(e)

        if not success:

            failed_pages.append(page)
            print(f"    [error] All {MAX_RETRIES} retries failed for '{anchor}' "
                  f"page {page} (last error: {last_error}). "
                  f"Skipping page and continuing...")


            page += 1
            next_url = f"https://api.osf.io/v2/preprints/?provider=socarxiv" \
                       f"&filter[description]={anchor}&page[size]=100&page={page}"
            url = next_url
            params = {}
            continue


        data = response.json()
        batch = data.get("data", [])
        results.extend(batch)
        print(f"    [api]   Page {page}: {len(batch)} papers (total: {len(results)})")

        if not batch:
            print(f"    [api] Empty page received — end of results for '{anchor}'")
            break

        url = data.get("links", {}).get("next")
        params = {}
        page += 1

    if failed_pages:
        print(f"    [warn] '{anchor}' completed with {len(results)} papers. "
              f"Failed pages (skipped): {failed_pages}")
    else:
        print(f"    [api] '{anchor}' completed cleanly. Total: {len(results)} papers")

    return results

# ── Step A: search and verify for a term list ─────────────────────────────────

def search_and_verify(term_list, label):
    """
    For each term:
      - get its anchor (last word if multi-word, itself if single)
      - fetch all papers via API using that anchor
      - for multi-word terms, verify the full exact phrase appears in the text
      - for single-word terms, the anchor fetch is already exact enough
    Returns: dict of { term -> set of paper IDs } and { paper_id -> paper object }
    """
    print(f"\n{'='*60}")
    print(f"Searching {label} terms...")
    print(f"{'='*60}")

    all_papers = {}
    term_to_ids = {}       

    anchor_to_terms = {}
    for term in term_list:
        anchor = get_anchor(term)
        anchor_to_terms.setdefault(anchor, []).append(term)

    print(f"\nAnchors to fetch: {list(anchor_to_terms.keys())}")

    for anchor, terms in anchor_to_terms.items():
        print(f"\n  Anchor '{anchor}' covers terms: {terms}")

        papers = fetch_by_anchor(anchor)
        print(f"  → {len(papers)} papers retrieved for anchor '{anchor}'")

        for term in terms:
            matched_ids = set()
            for paper in papers:
                pid = paper.get("id")
                text = get_searchable_text(paper)

                if is_multi_word(term):
                    if exact_match(text, term):
                        matched_ids.add(pid)
                        all_papers[pid] = paper

                else:
                    matched_ids.add(pid)
                    all_papers[pid] = paper

            term_to_ids[term] = matched_ids
            print(f"  → Term '{term}': {len(matched_ids)} papers after exact match check")

    return term_to_ids, all_papers

# ── Step B: pairwise LLM × HEALTH_OUTCOME matching ───────────────────────────

def pairwise_match(llm_term_to_ids, health_term_to_ids, all_papers):
    """
    For each LLM term, check against every HEALTH_OUTCOME term.
    Keep papers that appear in both sets.
    Returns list of result dicts.
    """
    print(f"\n{'='*60}")
    print(f"Pairwise matching: each LLM term × each HEALTH_OUTCOME term")
    print(f"{'='*60}")

    paper_results = {}

    for llm_term, llm_ids in llm_term_to_ids.items():
        for health_term, health_ids in health_term_to_ids.items():
            overlap = llm_ids & health_ids
            if overlap:
                print(f"  '{llm_term}' × '{health_term}': {len(overlap)} papers")
                for pid in overlap:
                    if pid not in paper_results:
                        paper_results[pid] = {
                            "paper": all_papers[pid],
                            "matched_llm": set(),
                            "matched_health": set(),
                            "pairs": [],
                        }
                    paper_results[pid]["matched_llm"].add(llm_term)
                    paper_results[pid]["matched_health"].add(health_term)
                    paper_results[pid]["pairs"].append((llm_term, health_term))
            else:
                print(f"  '{llm_term}' × '{health_term}': 0 papers")

    return paper_results

# ── Main ──────────────────────────────────────────────────────────────────────

print("=" * 60)
print("SocArXiv: LLM × Health Outcome search")
print("=" * 60)


llm_term_to_ids, llm_papers = search_and_verify(LLM_CHOICES, "LLM")
health_term_to_ids, health_papers = search_and_verify(HEALTH_OUTCOMES, "HEALTH_OUTCOME")

all_papers = {**llm_papers, **health_papers}

print(f"\nTotal unique papers found across LLM terms:    {len(llm_papers)}")
print(f"Total unique papers found across health terms:  {len(health_papers)}")

paper_results = pairwise_match(llm_term_to_ids, health_term_to_ids, all_papers)

print(f"\n→ Unique papers matching at least one LLM × health pair: {len(paper_results)}")


print(f"\n{'='*60}")
print(f"Final results ({len(paper_results)} papers):")
print(f"{'='*60}")

if not paper_results:
    print("\nNo papers matched.")
else:
    for i, (pid, result) in enumerate(paper_results.items(), 1):
        info = extract_info(result["paper"])
        contribs = get_contributors(pid)

        print(f"\n[{i}] {info['title']}")
        print(f"     ID:              {pid}")
        print(f"     Published:       {info['date_published']}")
        print(f"     Provider:        {info['provider']}")
        print(f"     URL:             {info['url']}")
        print(f"     DOI:             {info['doi'] or 'None'}")
        print(f"     Status:          {info['reviews_state']}")
        print(f"     Contributors:    {', '.join(contribs) if contribs else 'None found'}")
        print(f"     Tags:            {', '.join(info['tags']) if info['tags'] else 'None'}")
        print(f"     Subjects:        {'; '.join(info['subjects']) if info['subjects'] else 'None'}")
        print(f"     LLM terms:       {', '.join(sorted(result['matched_llm']))}")
        print(f"     Health terms:    {', '.join(sorted(result['matched_health']))}")
        print(f"     Pairs matched:")
        for llm_t, health_t in sorted(set(result["pairs"])):
            print(f"       - {llm_t} × {health_t}")
        desc = info["description"] or ""
        print(f"     Abstract:        {desc[:400]}..." if len(desc) > 400 else f"     Abstract:        {desc}")